# Oracle Detection: distance versus Linajea movement + ILP

Dùng **đúng tâm GT** của hai validation sample `6bba_78a7bd97`, `6bba_cf35214c` để loại ảnh hưởng của lỗi detection. Checkpoint lấy từ lần train 8 sample `44b6`; notebook **không train lại**. GEFF edge chỉ được dùng để chấm điểm sau khi bốn phương pháp đã nối xong.

Năm cách nối: bốn baseline khoảng cách/movement × greedy/ILP, cộng `linajea_oracle`. Bốn baseline dùng **cùng candidate edge set** trong bán kính vật lý 12 µm. `linajea_oracle` dùng chính `candidate_edges` (tìm parent quanh điểm do movement dự đoán, tối đa 3 parent) và `solve_basic_ilp` của fork Biohub, với tất cả GT node được ghim là đã chọn. Đây là phép thử oracle để xác định giá trị của linking; hai sample validation thuộc `train/` có GT centers, còn tập `test/` dùng để submission không cung cấp GT.

Gắn Biohub competition data và **Notebook Output Files** của lần chạy reverse làm Kaggle Input. Output mới được lưu ở `/kaggle/working/biohub_oracle_linking_reverse/`.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
import pandas as pd
from IPython.display import display

DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
OUTPUT_DIR = Path('/kaggle/working/biohub_oracle_linking_reverse')
REPO = Path('/kaggle/working/linajea')
PINNED_COMMIT = None  # Có thể điền SHA commit sau khi push code mới lên GitHub.
PREVIOUS_OUTPUT = None  # Hoặc Path('/kaggle/input/TEN-NOTEBOOK/biohub_holdout_8x2_reverse')
PREVIOUS_ORACLE_OUTPUT = None  # Tùy chọn: output của lần chạy oracle trước để tái dùng movement đã tính.
if PREVIOUS_OUTPUT is None:
    found = list(Path('/kaggle/input').glob('*/biohub_holdout_8x2_reverse/split.json'))
    if len(found) != 1:
        raise FileNotFoundError('Không tìm thấy duy nhất split.json của output reverse. Hãy gắn Notebook Output Files rồi đặt PREVIOUS_OUTPUT thủ công. Tìm thấy: ' + str(found))
    PREVIOUS_OUTPUT = found[0].parent
SPLIT = PREVIOUS_OUTPUT / 'split.json'
CONFIG = PREVIOUS_OUTPUT / 'train/config.toml'
CHECKPOINT = PREVIOUS_OUTPUT / 'train/train_net_checkpoint_16000'
assert DATA_DIR.is_dir(), DATA_DIR
for path in (SPLIT, CONFIG, CHECKPOINT):
    assert path.is_file(), f'Thiếu file: {path}'
split = json.loads(SPLIT.read_text())
assert set(split['validation']) == {'6bba_78a7bd97', '6bba_cf35214c'}, split['validation']
assert len(split['train']) == 8 and all(s.startswith('44b6_') for s in split['train'])
config_text = CONFIG.read_text()
assert all(s not in config_text for s in split['validation']), 'Validation GT xuất hiện trong train config'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if PREVIOUS_ORACLE_OUTPUT is None:
    old_oracle = list(Path('/kaggle/input').glob('*/biohub_oracle_linking_reverse/summary.csv'))
    if len(old_oracle) == 1:
        PREVIOUS_ORACLE_OUTPUT = old_oracle[0].parent
if PREVIOUS_ORACLE_OUTPUT is not None:
    for sample in split['validation']:
        target = OUTPUT_DIR / sample
        target.mkdir(parents=True, exist_ok=True)
        for filename in ('oracle_nodes_with_model.csv', 'oracle_cache_signature.json'):
            source = PREVIOUS_ORACLE_OUTPUT / sample / filename
            if source.is_file() and not (target / filename).exists():
                shutil.copy2(source, target / filename)
    print('Reusing oracle movement cache:', PREVIOUS_ORACLE_OUTPUT)
print('Output reverse:', PREVIOUS_OUTPUT)
print('Checkpoint:', CHECKPOINT, 'bytes:', CHECKPOINT.stat().st_size)
print('Validation:', split['validation'])


## Cài repo và dependency

Bật GPU trong Kaggle Settings nếu chưa có cache `oracle_nodes_with_model.csv`. Model được chạy **chỉ tại các tile chứa GT center** để lấy movement và score ở đúng voxel GT; không suy luận lại toàn bộ volume.

In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'biohub-io', '--single-branch',
                    'https://github.com/hathanhv/linajea.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'biohub-io'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout',
                PINNED_COMMIT or 'origin/biohub-io'], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert (REPO / 'run_scripts/biohub_oracle_linking.py').is_file(), 'Push code oracle-linking lên fork trước'
(OUTPUT_DIR / 'code_commit.txt').write_text(COMMIT + '\n')
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *map(str, args)], check=True)
pip('-r', REPO / 'requirements-biohub-io.txt')
pip('-r', REPO / 'requirements-biohub-train.txt')
pip('--no-deps', 'git+https://github.com/abred/daisy@ab7b82054126a50ab4ad5dcf2ef885b02d33aab8')
pip('--no-deps', 'git+https://github.com/Kainmueller-Lab/funlib.learn.torch@36ef666d16bddda0bf3146d4c0efbe9b95c8463d')
pip('-e', REPO, '--no-deps')
print('Linajea commit:', COMMIT)


## Chạy năm phương pháp trên cùng oracle nodes

Chi phí được giữ như experiment hiện tại: appearance = 7, division = 1, edge = 0,35 × lỗi khoảng cách. Node cost không ảnh hưởng lựa chọn edge vì **tất cả GT nodes được cố định là tồn tại**. `distance_ilp` so với `distance_greedy` đo tác động của tối ưu toàn cục; `movement_ilp` so với `distance_ilp` đo tác động của movement khi dùng chung tập cạnh. `linajea_oracle` kiểm tra cả bước tạo cạnh theo movement và basic ILP của fork.

`oracle_node_recall = 1` theo thiết kế và không phải chất lượng detector. `known_edge_recall` là tỷ lệ cạnh GT được khôi phục. Cạnh nối không có trong sparse GT chỉ được ghi là *unconfirmed*, không tự động gọi là sai.

In [ ]:
import torch
if not torch.cuda.is_available() and not all((OUTPUT_DIR / s / 'oracle_nodes_with_model.csv').is_file() for s in split['validation']):
    raise RuntimeError('Bật GPU để lấy movement tại GT centers')
cmd = [sys.executable, '-m', 'run_scripts.biohub_oracle_linking',
       '--data-dir', str(DATA_DIR), '--split', str(SPLIT),
       '--config', str(CONFIG), '--checkpoint', str(CHECKPOINT),
       '--output-dir', str(OUTPUT_DIR),
       '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
       '--max-distance-um', '12', '--appearance-cost', '7',
       '--division-cost', '1', '--edge-weight', '0.35']
subprocess.run(cmd, cwd=REPO, check=True)
results = pd.read_csv(OUTPUT_DIR / 'summary.csv')
display(results[['sample', 'method', 'oracle_nodes', 'candidate_edges',
                 'candidate_gt_edges', 'known_gt_edges', 'selected_edges',
                 'recovered_gt_edges', 'known_edge_recall',
                 'confirmed_edge_fraction_lower_bound', 'unconfirmed_edges']])


## Tổng hợp hai sample và hình so sánh

Bảng tổng hợp cộng số cạnh trước khi lấy tỷ lệ, không lấy trung bình đơn giản hai recall. Trần `candidate_edge_ceiling` cho biết bao nhiêu cạnh GT nằm trong bán kính 12 µm và vì thế bất kỳ phương pháp nào ở đây có thể nối được.

In [ ]:
import matplotlib.pyplot as plt
totals = results.groupby('method', as_index=False)[
    ['candidate_gt_edges', 'known_gt_edges', 'selected_edges',
     'recovered_gt_edges', 'unconfirmed_edges']].sum()
totals['known_edge_recall'] = totals['recovered_gt_edges'] / totals['known_gt_edges']
totals['candidate_edge_ceiling'] = totals['candidate_gt_edges'] / totals['known_gt_edges']
totals['confirmed_fraction_lower_bound'] = totals['recovered_gt_edges'] / totals['selected_edges']
totals.to_csv(OUTPUT_DIR / 'aggregate.csv', index=False)
display(totals[['method', 'recovered_gt_edges', 'known_gt_edges',
                'known_edge_recall', 'candidate_edge_ceiling',
                'selected_edges', 'unconfirmed_edges',
                'confirmed_fraction_lower_bound']])
original_summary = PREVIOUS_OUTPUT / 'validation/summary_7um.csv'
if original_summary.is_file():
    original = pd.read_csv(original_summary)
    print('Pipeline gốc (detection + linking; chỉ để tham chiếu):')
    display(original[['sample', 'known_node_recall', 'known_edge_recall']])
    print('Oracle dùng GT centers nên recall edge giữa hai bảng không phải so sánh công bằng về detection.')
ax = results.pivot(index='sample', columns='method', values='known_edge_recall').plot.bar(
    figsize=(10, 4), ylim=(0, 1), rot=0, ylabel='Known GT edge recall')
ax.set_title('Oracle centers: baselines and Linajea module')
ax.legend(title='Method', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'edge_recall_comparison.png', dpi=160, bbox_inches='tight')
plt.show()
print('Files:', OUTPUT_DIR / 'summary.csv', OUTPUT_DIR / 'aggregate.csv',
      OUTPUT_DIR / 'edge_recall_comparison.png', sep='\n')
